# Chapter 10 — Error Handling in Multi-agent Systems · Practice

**Goal:** make failures visible and actionable at every layer of an OpenAI multi-agent system: classify `openai` errors, control the client's and the runner's retries, turn tool errors into data, return structured worker results with partial findings, and finish with a research coordinator that survives injected faults and writes a coverage-annotated report.

- Theory: [README.md](README.md) (sections 10.1–10.10). This notebook holds the code.
- **Estimated time:** 3–4 hours
- **Estimated API cost:** under $0.05 with the default `gpt-6-luna` (four small live cells). Every section runs offline first, on a mock HTTP layer or the Agents SDK's `ScriptedModel`, so a learner without credit can do the whole chapter.
- **Prerequisites:** [Chapter 03](../03-agents-sdk-and-agents-api/README.md) (Runner, agents as tools, guardrails, tracing), [Chapter 09](../09-escalation-human-in-the-loop/README.md), the repo venv from [00-prerequisites](../../00-prerequisites/README.md), and `OPENAI_API_KEY` in `.env` for the live cells.

Cells marked **(live)** call the OpenAI API. Everything else is offline. Two offline tools do the heavy lifting:

- a **real `OpenAI` client whose HTTP layer is a local function** (`httpx2.MockTransport`), so you see the SDK's actual exceptions and actual retry behavior without a network;
- the Agents SDK's own test double, **`agents.testing.ScriptedModel`**, which replays model steps, including injected model-call failures (`ModelStep.raise_error(...)`).

In [ ]:
import asyncio
import json
import os
import random
import time
from collections import Counter
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Any, Literal

import httpx2
import openai
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from pydantic import BaseModel

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    MaxTurnsExceeded,
    ModelBehaviorError,
    ModelRefusalError,
    ModelRetrySettings,
    ModelSettings,
    ModelTimeoutError,
    OutputGuardrailTripwireTriggered,
    RunConfig,
    RunContextWrapper,
    RunErrorHandlerInput,
    RunErrorHandlerResult,
    RunResult,
    Runner,
    SpanError,
    ToolInputGuardrailTripwireTriggered,
    ToolOutputGuardrailTripwireTriggered,
    ToolTimeoutError,
    UserError,
    custom_span,
    function_tool,
    input_guardrail,
    output_guardrail,
    retry_policies,
    set_trace_processors,
    trace,
)
from agents.testing import ModelStep, ScriptedModel, assistant_message, function_call
from agents.tracing import TracingProcessor
from agents.tracing.processors import default_processor

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
OFFLINE = RunConfig(tracing_disabled=True)   # offline agent runs: nothing is exported
print("Model:", MODEL, "| openai", openai.__version__)

In [ ]:
# Live cells need a key. Without one, client is None and the live cells print a skip note.
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None   # defaults: max_retries=2, timeout 600 s read / 5 s connect
print("Live client ready." if client else "No OPENAI_API_KEY: live cells will be skipped; offline cells still run.")

Shared helpers for the offline cells. `mock_client(handler)` is a normal `OpenAI` client: the same retry code, the same exception classes, the same `request_id` handling. Only the transport is replaced, so `handler(request)` decides what the "server" answers.

In [ ]:
def mock_client(handler, **kwargs) -> OpenAI:
    """A real OpenAI client whose HTTP layer is a local function. No network, no key needed."""
    transport = httpx2.MockTransport(handler)
    return OpenAI(api_key="sk-offline-demo", http_client=httpx2.Client(transport=transport), **kwargs)


def error_response(status: int, code: str | None = None, type_: str | None = None, message: str = "",
                   headers: dict | None = None, param: str | None = None) -> httpx2.Response:
    """An HTTP error in the API's error envelope: {"error": {message, type, code, param}}."""
    body = {"error": {"message": message or f"HTTP {status} ({code})", "type": type_, "code": code, "param": param}}
    return httpx2.Response(status, json=body, headers={"x-request-id": f"req_{status}_{code or 'none'}", **(headers or {})})


def ok_response(text: str = "OK", **fields) -> httpx2.Response:
    body = {"id": "resp_mock", "object": "response", "created_at": 0, "model": MODEL, "status": "completed",
            "output": [{"type": "message", "id": "msg_1", "role": "assistant", "status": "completed",
                        "content": [{"type": "output_text", "text": text, "annotations": []}]}],
            "parallel_tool_calls": True, "tool_choice": "auto", "tools": [], **fields}
    return httpx2.Response(200, json=body, headers={"x-request-id": "req_ok_1"})


def read_timeout(request):
    raise httpx2.ReadTimeout("read timed out", request=request)


def connect_failure(request):
    raise httpx2.ConnectError("DNS lookup failed", request=request)


def raise_from(handler) -> openai.APIError:
    """Send one request (no retries) to a mock server and return the exception the SDK raises."""
    try:
        mock_client(handler, max_retries=0).responses.create(model=MODEL, input="ping")
    except openai.APIError as exc:
        return exc
    raise AssertionError("the mock server answered successfully")


# The Agents SDK side: the same exception objects can be injected into a ScriptedModel.
def overloaded_503() -> openai.InternalServerError:
    return raise_from(lambda r: error_response(503, "server_is_overloaded", "service_unavailable_error",
                                               "The model is temporarily overloaded."))


def billing_429() -> openai.RateLimitError:
    return raise_from(lambda r: error_response(429, "credit_balance_exhausted", message="No prepaid credits remaining."))


exc = overloaded_503()
print(type(exc).__name__, exc.status_code, exc.code, exc.type, exc.request_id)

## 10.1 Error categories on the OpenAI platform

Four categories decide the response: **transient** (retry, bounded), **validation** (fix the input), **business** (explain or route, never loop), **permission** (escalate). A **valid empty result** is not an error at all. On OpenAI, the signal is spread over the exception class, the HTTP status and `error.code`, and the same `429` can mean "slow down" or "no credits left". [Theory](README.md#101-error-categories-on-the-openai-platform)

In [ ]:
NON_RETRYABLE_429 = {"credit_balance_exhausted", "organization_spend_limit_exceeded",
                     "project_spend_limit_exceeded", "organization_usage_limit_exceeded"}


@dataclass
class ErrorInfo:
    category: Literal["transient", "validation", "business", "permission", "unknown"]
    retryable: bool
    detail: str


def classify_openai_error(exc: Exception) -> ErrorInfo:
    """Map an openai exception to a category. Called AFTER the client's own retries."""
    if isinstance(exc, openai.APITimeoutError):            # check before APIConnectionError (subclass)
        return ErrorInfo("transient", True, "timeout")
    if isinstance(exc, openai.APIConnectionError):
        return ErrorInfo("transient", True, "connection")
    if isinstance(exc, openai.RateLimitError):
        if exc.code in NON_RETRYABLE_429:                   # billing / quota: a human must act
            return ErrorInfo("business", False, exc.code)
        return ErrorInfo("transient", True, exc.code or "rate_limited")
    if isinstance(exc, openai.InternalServerError):         # 500, 503 server_is_overloaded, ...
        return ErrorInfo("transient", True, f"{exc.status_code} {exc.code}")
    if isinstance(exc, (openai.AuthenticationError, openai.PermissionDeniedError)):
        return ErrorInfo("permission", False, exc.code or str(exc.status_code))
    if isinstance(exc, (openai.BadRequestError, openai.UnprocessableEntityError, openai.NotFoundError)):
        return ErrorInfo("validation", False, f"param={exc.param}")
    return ErrorInfo("unknown", False, type(exc).__name__)


# Error bodies: codes and types follow the error-codes guide where it documents them; others are left None.
SAMPLES = {
    "network timeout": read_timeout,
    "connection failure": connect_failure,
    "429 slow_down": lambda r: error_response(429, "slow_down", "rate_limit_error", headers={"retry-after": "2"}),
    "429 rate limit (no code)": lambda r: error_response(429, None, None, "Rate limit reached for requests"),
    "429 credit_balance_exhausted": lambda r: error_response(429, "credit_balance_exhausted"),
    "429 project_spend_limit": lambda r: error_response(429, "project_spend_limit_exceeded"),
    "503 server_is_overloaded": lambda r: error_response(503, "server_is_overloaded", "service_unavailable_error"),
    "500 internal": lambda r: error_response(500, None, None, "The server had an error"),
    "400 bad parameter": lambda r: error_response(400, None, "invalid_request_error", "Invalid value", param="max_output_tokens"),
    "401 bad key": lambda r: error_response(401, None, None, "Incorrect API key provided"),
    "403 no model access": lambda r: error_response(403, None, None, "Project does not have access to this model"),
}

print(f"{'scenario':<30}{'exception':<24}{'status':>6}  {'category':<11}{'retry':<6}detail")
for label, handler in SAMPLES.items():
    exc = raise_from(handler)
    info = classify_openai_error(exc)
    print(f"{label:<30}{type(exc).__name__:<24}{str(getattr(exc, 'status_code', '-')):>6}  "
          f"{info.category:<11}{str(info.retryable):<6}{info.detail}")

> **Observe:** three `RateLimitError`s, three different decisions. The exception class alone would have put the empty wallet in the "back off and retry" bucket. `exc.code` is what separates them.

The second trap is **empty versus failed**. Below, two versions of the same search tool see the same two situations: a search that ran and found nothing, and a search that timed out.

In [ ]:
class FakeArchive:
    def __init__(self, down: bool):
        self.down = down

    def search(self, topic: str) -> list[dict]:
        if self.down:
            raise TimeoutError("archive did not answer within 20 s")
        return []                                   # ran correctly, nothing matched


def naive_search(archive: FakeArchive, topic: str) -> str:
    try:
        return json.dumps(archive.search(topic))
    except Exception:
        return "[]"                                 # silent suppression


def honest_search(archive: FakeArchive, topic: str) -> str:
    try:
        hits = archive.search(topic)
        return json.dumps({"status": "ok", "results": hits, "note": "searched, 0 matches" if not hits else None})
    except TimeoutError as exc:
        return json.dumps({"status": "error", "category": "transient", "retryable": True,
                           "message": str(exc), "attempted_query": topic})


def coordinator_sentence(tool_output: str) -> str:
    data = json.loads(tool_output)
    if isinstance(data, list) or data.get("status") == "ok":
        results = data if isinstance(data, list) else data["results"]
        return "No publications found." if not results else f"{len(results)} publications found."
    return f"Could not search ({data['category']}, retryable={data['retryable']}): section NOT COVERED."


for name, archive in {"valid empty": FakeArchive(down=False), "archive down": FakeArchive(down=True)}.items():
    print(f"{name:<13} naive  -> {coordinator_sentence(naive_search(archive, 'AI in music')):<45}"
          f"honest -> {coordinator_sentence(honest_search(archive, 'AI in music'))}")

> **Observe:** with the naive tool, the coordinator writes "No publications found." in both cases. One of them is a lie, and nothing downstream can detect it. The honest tool costs one extra field.
>
> **Try it:** add a third situation, a `PermissionError("403: subscription expired")`, to `FakeArchive` and `honest_search`. Which category and `retryable` value should it report?

## 10.2 OpenAI client errors, retries and timeouts

Every `OpenAI` client already retries connection errors, timeouts, `408`, `409`, `429` and `≥500` twice (three attempts), with exponential backoff, jitter, `Retry-After` up to 120 s, and the `x-should-retry` header as an override. Your code only sees an exception **after** those retries, so design the next layer with that in mind. [Theory](README.md#102-openai-client-errors-retries-and-timeouts)

In [ ]:
class ScriptedServer:
    """Mock HTTP server: plays a script of handlers in order, then repeats the last one. Counts requests."""

    def __init__(self, *script):
        self.script, self.requests = list(script), 0

    def __call__(self, request):
        self.requests += 1
        return self.script[min(self.requests, len(self.script)) - 1](request)


# The SDK also reads `retry-after-ms`; we use it to keep the demo fast. Real servers usually send `retry-after`.
FAST = {"retry-after-ms": "20"}
SCENARIOS = {
    "429 slow_down x2, then OK": [lambda r: error_response(429, "slow_down", "rate_limit_error", headers=FAST),
                                  lambda r: error_response(429, "slow_down", "rate_limit_error", headers=FAST),
                                  lambda r: ok_response("recovered")],
    "429 credit_balance_exhausted": [lambda r: error_response(429, "credit_balance_exhausted", headers=FAST)],
    "same, server sends x-should-retry: false": [lambda r: error_response(429, "credit_balance_exhausted",
                                                                          headers={"x-should-retry": "false"})],
    "503 overloaded, Retry-After: 300": [lambda r: error_response(503, "server_is_overloaded", headers={"retry-after": "300"})],
    "400 bad parameter": [lambda r: error_response(400, None, "invalid_request_error", param="tools")],
    "read timeout (no headers: backoff)": [read_timeout],
}

print(f"{'scenario':<42}{'requests':>8}  {'time':>6}  outcome")
for label, script in SCENARIOS.items():
    server = ScriptedServer(*script)
    start = time.monotonic()
    try:
        resp = mock_client(server).responses.create(model=MODEL, input="ping")
        outcome = f"success: {resp.output_text!r} (request id {resp._request_id})"
    except openai.APIError as exc:
        outcome = f"{type(exc).__name__} code={exc.code} request_id={getattr(exc, 'request_id', None)}"
    print(f"{label:<42}{server.requests:>8}  {time.monotonic() - start:5.2f}s  {outcome}")

# Per-call override: one attempt only.
server = ScriptedServer(lambda r: error_response(429, "slow_down", "rate_limit_error", headers=FAST))
try:
    mock_client(server).with_options(max_retries=0, timeout=20).responses.create(model=MODEL, input="ping")
except openai.RateLimitError:
    print(f"\nwith_options(max_retries=0): {server.requests} request")

> **Observe:** the billing `429` was sent **three times** before your code saw it, because the SDK's rule is status-based. Only the server's `x-should-retry: false` stops that. A `Retry-After` above 120 s makes the SDK give up immediately and hand the decision to you. The timeout row took about 1.1–1.5 s: that is the SDK's backoff (0.5 s, then 1 s, minus up to 25% jitter).

Now stack a typical app-level retry decorator on top and count what one logical call costs during an outage.

In [ ]:
def naive_retry(times: int):
    """The anti-pattern: retry every RateLimitError / InternalServerError, regardless of code."""
    def wrap(fn):
        def inner(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except (openai.RateLimitError, openai.InternalServerError):
                    if attempt == times:
                        raise
        return inner
    return wrap


def count_requests(handler, *, client_retries: int, app_retries: int, worker_reruns: int) -> int:
    server = ScriptedServer(handler)
    llm = mock_client(server, max_retries=client_retries)

    @naive_retry(app_retries)
    def call():
        return llm.responses.create(model=MODEL, input="summarize")

    for _ in range(worker_reruns):          # the worker re-runs its whole step when it fails
        try:
            call()
            break
        except openai.APIError:
            pass
    return server.requests


outage = lambda r: error_response(503, "server_is_overloaded", "service_unavailable_error", headers=FAST)
wallet = lambda r: error_response(429, "credit_balance_exhausted", headers=FAST)
print("503 outage, client 2 retries x app 3 tries x worker 2 runs :",
      count_requests(outage, client_retries=2, app_retries=3, worker_reruns=2), "requests for ONE step")
print("empty wallet, same stack                                 :",
      count_requests(wallet, client_retries=2, app_retries=3, worker_reruns=2), "requests for ONE step")


def owned_retry_call(llm: OpenAI, *, attempts: int = 3, deadline_s: float = 10.0):
    """One owning layer: client retries off, classify, retry only transient, bounded by attempts AND time."""
    deadline = time.monotonic() + deadline_s
    for attempt in range(1, attempts + 1):
        remaining = deadline - time.monotonic()
        try:   # each attempt gets only the time that is left (the client's default read timeout is 600 s)
            return llm.with_options(max_retries=0, timeout=remaining).responses.create(model=MODEL, input="summarize")
        except openai.APIError as exc:
            info = classify_openai_error(exc)
            delay = min(0.01 * 2 ** attempt, 0.1)        # tiny delays for the demo; honor Retry-After in production
            if not info.retryable or attempt == attempts or time.monotonic() + delay >= deadline:
                raise                                    # no sleep, and no attempt, that would cross the deadline
            time.sleep(delay)


for name, handler in {"503 outage": outage, "empty wallet": wallet}.items():
    server = ScriptedServer(handler)
    try:
        owned_retry_call(mock_client(server))
    except openai.APIError as exc:
        print(f"one owning layer, {name:<13}: {server.requests} request(s), then {classify_openai_error(exc)}")

> **Observe:** 18 requests for one step, and the wallet case is no better than the outage. With one owning layer the outage costs 3 requests and the empty wallet costs 1.

`429 slow_down` measures how fast your traffic **grew**. A coordinator that starts 50 workers at once on a cold project is the classic trigger, so cap concurrency instead of retrying harder:

In [ ]:
async def fan_out(n_workers: int, limit: int | None) -> int:
    """Run n fake model calls; return the peak number in flight."""
    gate = asyncio.Semaphore(limit) if limit else None
    in_flight = peak = 0

    async def one_call():
        nonlocal in_flight, peak
        in_flight += 1
        peak = max(peak, in_flight)
        await asyncio.sleep(0.01)
        in_flight -= 1

    async def guarded():
        if gate is None:
            return await one_call()
        async with gate:
            return await one_call()

    await asyncio.gather(*(guarded() for _ in range(n_workers)))
    return peak


print("peak concurrent calls, no cap :", await fan_out(50, None))
print("peak concurrent calls, cap 8  :", await fan_out(50, 8))

In [ ]:
# (live) ~ $0.001. A successful call: keep its request id for later investigation.
if not os.getenv("OPENAI_API_KEY"):
    print("Skipping (live): set OPENAI_API_KEY in .env")
else:
    resp = client.with_options(max_retries=1, timeout=30).responses.create(
        model=MODEL, input="Reply with the single word OK.", max_output_tokens=400)
    print("status:", resp.status, "| text:", resp.output_text, "| request id:", resp._request_id)

    # (live) a deliberately invalid request: no retries happen for a 4xx validation error.
    try:
        client.responses.create(model="no-such-model-for-ch10", input="ping", max_output_tokens=400)
    except openai.APIStatusError as exc:
        print(type(exc).__name__, exc.status_code, "code:", exc.code, "| request id:", exc.request_id)
        print("classified as:", classify_openai_error(exc))

> **Try it:** in `owned_retry_call`, read `exc.response.headers.get("retry-after")` and sleep at least that long (capped by the remaining deadline). Then rerun the `503 outage` case with `"retry-after": "1"` in the headers.

## 10.3 Agents SDK exceptions and run-level handlers

Every Agents SDK exception inherits from `AgentsException`; when the runner fails mid-run it fills `run_data` (input, `new_items`, raw responses, last agent; check for `None`), your source of partial results. `openai.APIError`s from the model call escape unwrapped. Three run-level controls change what happens before an exception escapes: `error_handlers`, `ModelSettings(retry=ModelRetrySettings(...))`, and `RunConfig(tool_not_found_behavior=...)`. [Theory](README.md#103-agents-sdk-exceptions-and-run-level-handlers)

In [ ]:
@function_tool
def search_notes(query: str) -> str:
    """Search the research notes.

    Args:
        query: Search terms.
    """
    return json.dumps({"status": "ok", "results": [{"title": f"Note about {query}", "source": "notes#1"}]})


def looping_model(n: int = 6) -> ScriptedModel:
    """A model that never stops calling the tool (each call needs a unique call_id)."""
    return ScriptedModel([[function_call("search_notes", {"query": f"angle {i}"}, call_id=f"loop_{i}")]
                          for i in range(n)])


# 1) MaxTurnsExceeded: the exception still carries what the tools returned.
try:
    await Runner.run(Agent(name="Looper", model=looping_model(), tools=[search_notes]), "Research AI music",
                     max_turns=3, run_config=OFFLINE)
except MaxTurnsExceeded as exc:
    salvaged = [i.output for i in exc.run_data.new_items if i.type == "tool_call_output_item"]
    print(f"MaxTurnsExceeded: {exc.message!r}; salvaged {len(salvaged)} tool outputs")
    print("  first:", salvaged[0])


# 2) error_handlers: the same failure becomes a controlled final output.
def on_max_turns(data: RunErrorHandlerInput) -> RunErrorHandlerResult:
    n_outputs = sum(i.type == "tool_call_output_item" for i in data.run_data.new_items)
    return RunErrorHandlerResult(
        final_output=json.dumps({"status": "partial_failure", "failure_type": "max_turns", "tool_outputs": n_outputs}),
        include_in_history=False,
    )


result = await Runner.run(Agent(name="Looper", model=looping_model(), tools=[search_notes]), "Research AI music",
                          max_turns=3, run_config=OFFLINE, error_handlers={"max_turns": on_max_turns})
print("\nerror_handlers['max_turns'] ->", result.final_output)

Next, runner-managed retries. They are **opt-in**. The step below raises a real `openai.InternalServerError` (`503 server_is_overloaded`) on the first model call; with a retry policy, the runner calls the model again and the run succeeds.

In [ ]:
TRANSIENT_POLICY = retry_policies.any(
    retry_policies.provider_suggested(),          # follow the provider's retry advice
    retry_policies.network_error(),
    retry_policies.http_status([408, 429, 500, 502, 503, 504]),
)
FAST_BACKOFF = {"initial_delay": 0.01, "max_delay": 0.05, "multiplier": 2.0, "jitter": False}   # demo speed


def agent_with(model: ScriptedModel, retry: ModelRetrySettings | None = None) -> Agent:
    return Agent(name="Worker", instructions="Answer briefly.", model=model,
                 model_settings=ModelSettings(retry=retry) if retry else ModelSettings())


# Without a retry setting: the 503 escapes unwrapped.
model = ScriptedModel([ModelStep.raise_error(overloaded_503()), [assistant_message("recovered")]])
try:
    await Runner.run(agent_with(model), "hi", run_config=OFFLINE)
except openai.InternalServerError as exc:
    print(f"no retry setting  -> {type(exc).__name__} {exc.code} after {len(model.calls)} model call")

# With ModelRetrySettings: one retry, success.
model = ScriptedModel([ModelStep.raise_error(overloaded_503()), [assistant_message("recovered")]])
result = await Runner.run(agent_with(model, ModelRetrySettings(max_retries=2, backoff=FAST_BACKOFF,
                                                               policy=TRANSIENT_POLICY)), "hi", run_config=OFFLINE)
print(f"with retry policy -> {result.final_output!r} after {len(model.calls)} model calls")

# The trap: a status-based policy retries the billing 429 too.
model = ScriptedModel([ModelStep.raise_error(billing_429()), [assistant_message("this should never happen")]])
result = await Runner.run(agent_with(model, ModelRetrySettings(max_retries=2, backoff=FAST_BACKOFF,
                                                               policy=retry_policies.http_status([429, 503]))),
                          "hi", run_config=OFFLINE)
print(f"http_status([429]) on credit_balance_exhausted -> retried ({len(model.calls)} calls)")


# The fix: a custom policy that vetoes non-retryable codes. A policy receives a RetryPolicyContext.
def not_billing(ctx) -> bool:
    return ctx.normalized.error_code not in NON_RETRYABLE_429


# The README 10.3 pattern: every sub-policy must agree, so the veto wins for billing codes.
SAFE_POLICY = retry_policies.all(TRANSIENT_POLICY, not_billing)
for label, err in {"credit_balance_exhausted": billing_429(),
                   "slow_down": raise_from(lambda r: error_response(429, "slow_down", "rate_limit_error"))}.items():
    model = ScriptedModel([ModelStep.raise_error(err), [assistant_message("recovered")]])
    try:
        result = await Runner.run(agent_with(model, ModelRetrySettings(max_retries=2, backoff=FAST_BACKOFF,
                                                                       policy=SAFE_POLICY)), "hi", run_config=OFFLINE)
        print(f"all(TRANSIENT_POLICY, not_billing) on {label:<25} -> {result.final_output!r} ({len(model.calls)} calls)")
    except openai.RateLimitError as exc:
        print(f"all(TRANSIENT_POLICY, not_billing) on {label:<25} -> raised {exc.code} ({len(model.calls)} call)")

> **Observe:** `retry_policies.http_status([429])` is status-based, exactly like the client. With the real OpenAI model, `provider_suggested()` also suggests retrying any `429` unless the server sends `x-should-retry: false` (verified in the 0.23.1 source). Put the "never retry billing" rule in your own policy, as `not_billing` does. `SAFE_POLICY` follows the README 10.3 pattern: `all(transient, not_billing)`. Here the transient part also includes `http_status([...])`, because `ScriptedModel` gives no provider advice; with the real OpenAI model, `provider_suggested()` alone covers `429`/`5xx`. So `slow_down` still recovers while the billing code is raised after one model call (with a real client, that one call still includes the client's own 2 retries).

The remaining runner exceptions, each triggered offline:

In [ ]:
from openai.types.responses import ResponseOutputMessage, ResponseOutputRefusal


class Answer(BaseModel):
    text: str


refusal = ResponseOutputMessage(id="msg_r", type="message", role="assistant", status="completed",
                                content=[ResponseOutputRefusal(type="refusal", refusal="I can't help with that.")])


async def slow_responder(call):
    await asyncio.sleep(1)
    return [assistant_message("too late")]


CASES = {
    "unknown tool (default)": (ScriptedModel([[function_call("lookup_order", {"id": 1}, call_id="c1")],
                                              [assistant_message("ok")]]), {}, OFFLINE),
    "unknown tool (return_error_to_model)": (
        ScriptedModel([[function_call("lookup_order", {"id": 1}, call_id="c1")], [assistant_message("Recovered.")]]),
        {}, RunConfig(tracing_disabled=True, tool_not_found_behavior="return_error_to_model")),
    "refusal": (ScriptedModel([[refusal]]), {"output_type": Answer}, OFFLINE),
    "model call too slow": (ScriptedModel([ModelStep.respond(slow_responder)]),
                            {"model_settings": ModelSettings(timeout=0.1)}, OFFLINE),
}
for label, (model, agent_kwargs, run_config) in CASES.items():
    agent = Agent(name="Worker", model=model, tools=[search_notes], **agent_kwargs)
    try:
        result = await Runner.run(agent, "go", run_config=run_config)
        outputs = [i.output for i in result.new_items if i.type == "tool_call_output_item"]
        print(f"{label:<38} -> completed; tool outputs {outputs}; final {result.final_output!r}")
    except (ModelBehaviorError, ModelRefusalError, ModelTimeoutError) as exc:
        detail = getattr(exc, "refusal", None) or getattr(exc, "timeout_seconds", None) or exc.message
        print(f"{label:<38} -> {type(exc).__name__}: {detail}")

> **Observe:** `tool_not_found_behavior="return_error_to_model"` turns a fatal `ModelBehaviorError` into the tool output `"Tool 'lookup_order' not found."`, which a real model can correct on the next turn.
>
> **Try it:** add `error_handlers={"model_refusal": lambda d: {"final_output": Answer(text="[refused]")}}` to the refusal case (pass it to `Runner.run`) and check that the run now completes with a typed output.

## 10.4 Tool errors as data

`function_call_output` has no error flag: the model learns that a tool failed only from the text in `output`. In the Agents SDK that text comes from the tool's `failure_error_function`. The default is a fixed, generic sentence; `None` makes the run fail; your own function returns a structured envelope. [Theory](README.md#104-tool-errors-as-data)

In [ ]:
class ToolFailure(Exception):
    """Raise this from tools for failures you anticipated."""

    def __init__(self, category: str, message: str, retryable: bool, suggestion: str | None = None):
        super().__init__(message)
        self.category, self.retryable, self.suggestion = category, retryable, suggestion


def tool_error_as_data(ctx: RunContextWrapper, error: Exception) -> str:
    if isinstance(error, ToolFailure):
        payload = {"category": error.category, "retryable": error.retryable,
                   "message": str(error), "suggestion": error.suggestion}
    elif isinstance(error, TimeoutError):
        payload = {"category": "transient", "retryable": True, "message": "Upstream search timed out.",
                   "suggestion": "Retry once with a narrower query, then report partial coverage."}
    else:                                                     # unexpected: don't leak internals
        payload = {"category": "unknown", "retryable": False,
                   "message": f"{type(error).__name__} in {getattr(ctx, 'tool_name', 'tool')}",
                   "suggestion": "Do not retry. Report this source as not covered."}
    return json.dumps({"status": "error", **payload})


def leaky_formatter(ctx, error: Exception) -> str:
    return f"Tool failed: {error!r}"                          # anti-pattern: raw exception text to the model


def make_archive_tool(**decorator_kwargs):
    @function_tool(**decorator_kwargs)
    async def search_archive(query: str, topic: str) -> str:
        """Search the research archive.

        Args:
            query: Search terms.
            topic: One of visual_art, music, literature.
        """
        if topic not in {"visual_art", "music", "literature"}:
            raise ToolFailure("validation", f"Unknown topic {topic!r}.", False, "Use visual_art, music or literature.")
        if query == "hang":
            await asyncio.sleep(2)                            # simulates a hung upstream call
        raise TimeoutError("upstream 504 from archive-internal.corp:8443 (db=prod-research)")
    return search_archive


async def tool_output_for(tool, arguments: dict) -> str:
    model = ScriptedModel([[function_call("search_archive", arguments, call_id="call_1")], [assistant_message("done")]])
    result = await Runner.run(Agent(name="Researcher", model=model, tools=[tool]), "go", run_config=OFFLINE)
    return next(i.output for i in result.new_items if i.type == "tool_call_output_item")


ARGS = {"query": "AI vocal synthesis", "topic": "music"}
print("default        :", await tool_output_for(make_archive_tool(), ARGS))
print("leaky          :", await tool_output_for(make_archive_tool(failure_error_function=leaky_formatter), ARGS))
print("as data        :", await tool_output_for(make_archive_tool(failure_error_function=tool_error_as_data), ARGS))
print("as data (val.) :", await tool_output_for(make_archive_tool(failure_error_function=tool_error_as_data),
                                                {"query": "x", "topic": "sculpture"}))
try:
    await tool_output_for(make_archive_tool(failure_error_function=None), ARGS)
except UserError as exc:
    print("None           : run failed with UserError:", exc.message[:70], "...")

> **Observe:** the default hides the hostname (good) but gives the model nothing to act on and says "Please try again" even for a validation error. The leaky formatter tells the model your internal hostname and database name. The structured envelope gives category, retryable flag and a next step, and nothing internal.

Tool timeouts work only on `async` tools. `timeout_behavior="error_as_result"` (the default) returns a message to the model; `"raise_exception"` fails the run with `ToolTimeoutError`.

In [ ]:
HANG = {"query": "hang", "topic": "music"}
print("error_as_result      :", await tool_output_for(make_archive_tool(timeout=0.2), HANG))
print("timeout_error_function:", await tool_output_for(make_archive_tool(
    timeout=0.2, timeout_error_function=lambda ctx, err: json.dumps(
        {"status": "error", "category": "transient", "retryable": True, "message": "Archive timed out after 0.2 s."})),
    HANG))
try:
    await tool_output_for(make_archive_tool(timeout=0.2, timeout_behavior="raise_exception"), HANG)
except ToolTimeoutError as exc:
    print("raise_exception      : ToolTimeoutError", exc.tool_name, exc.timeout_seconds)

try:
    @function_tool(timeout=5)
    def sync_lookup(order_id: str) -> str:
        """Look up an order."""
        return order_id
except ValueError as exc:
    print("sync tool + timeout  : ValueError:", exc)

> **Try it:** make `tool_error_as_data` also log the full exception (`logging.exception`) before returning the safe payload, so the details reach your logs and the trace but not the model.

## 10.5 Structured worker results with partial results

A tool error informs the **worker's** model. A `WorkerResult` informs the **coordinator**. Make it an `output_type`, and wrap every worker run so that the coordinator gets a `WorkerResult` even when the runner raises. [Theory](README.md#105-structured-worker-results-with-partial-results)

In [ ]:
class Finding(BaseModel):
    title: str
    source: str


class WorkerResult(BaseModel):                     # strict-schema friendly: every field required
    topic: str
    status: Literal["success", "partial_failure", "failure"]
    failure_type: Literal["timeout", "rate_limited", "validation", "permission",
                          "policy_blocked", "max_turns", "unknown"] | None
    retryable: bool
    attempted_query: str
    findings: list[Finding]                        # partial results survive a failure
    alternative_approaches: list[str]              # what the coordinator could try instead
    coverage_impact: str | None                    # "Not covered: licensed sales data"


# When several failures meet in one result, the most actionable one wins, so a permission or policy failure
# is never hidden behind a timeout that happened to come first.
FAILURE_PRECEDENCE = ["policy_blocked", "permission", "rate_limited", "validation", "max_turns", "unknown", "timeout"]


def most_actionable(failure_types) -> str | None:
    types = [t for t in failure_types if t]
    return min(types, key=FAILURE_PRECEDENCE.index) if types else None


def parse_findings(tool_outputs: list[str]) -> list[Finding]:
    """Turn successful tool outputs (JSON with status ok) into Findings; ignore errors and non-JSON."""
    findings = []
    for raw in tool_outputs:
        try:
            data = json.loads(raw)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(data, dict) and data.get("status") == "ok":
            findings += [Finding(**hit) for hit in data.get("results") or []]
    return findings


def salvage(exc: Exception) -> list[Finding]:
    """Every AgentsException carries run_data: keep the findings the tools returned before the failure."""
    run_data = getattr(exc, "run_data", None)
    return parse_findings([i.output for i in (run_data.new_items if run_data else [])
                           if i.type == "tool_call_output_item"])


def failed(topic: str, task: str, failure_type: str, retryable: bool, impact: str, exc: Exception,
           alternatives: list[str] | None = None) -> WorkerResult:
    findings = salvage(exc)
    return WorkerResult(topic=topic, status="partial_failure" if findings else "failure", failure_type=failure_type,
                        retryable=retryable, attempted_query=task, findings=findings,
                        alternative_approaches=alternatives or [], coverage_impact=impact)


async def run_worker(agent: Agent, topic: str, task: str, *, run_config: RunConfig | None = None) -> WorkerResult:
    """Always returns a WorkerResult for the documented failures; the coordinator catches anything else."""
    try:
        result = await Runner.run(agent, task, max_turns=6, run_config=run_config)
        return result.final_output_as(WorkerResult)
    except MaxTurnsExceeded as exc:
        return failed(topic, task, "max_turns", False, "Worker stopped at its turn limit; section incomplete.", exc,
                      ["Split the topic into narrower sub-questions"])
    except (InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered,
            ToolInputGuardrailTripwireTriggered, ToolOutputGuardrailTripwireTriggered) as exc:
        return failed(topic, task, "policy_blocked", False, "Blocked by policy; needs human review.", exc)
    except ModelRefusalError as exc:                                      # don't rephrase around a refusal
        return failed(topic, task, "policy_blocked", False, "The model refused; route to review.", exc)
    except (ModelTimeoutError, ToolTimeoutError) as exc:                  # AgentsException, NOT TimeoutError
        return failed(topic, task, "timeout", True, f"{type(exc).__name__}: section incomplete.", exc,
                      ["Retry once with a narrower task"])
    except ModelBehaviorError as exc:                                     # unknown tool, bad JSON, bad schema
        return failed(topic, task, "validation", False, "Model output broke the tool/schema contract; fix the agent.", exc)
    except openai.APIError as exc:                                       # after client/runner retries
        info = classify_openai_error(exc)
        failure_type = ("rate_limited" if isinstance(exc, openai.RateLimitError)       # incl. billing 429s
                        else "timeout" if isinstance(exc, openai.APITimeoutError)
                        else info.category if info.category in ("permission", "validation")
                        else "unknown")
        return failed(topic, task, failure_type, info.retryable,
                      f"Model call failed ({info.category}: {info.detail}).", exc)

In [ ]:
@input_guardrail(run_in_parallel=False)           # blocking: runs before the model is called
def no_personal_data(ctx, agent, user_input) -> GuardrailFunctionOutput:
    return GuardrailFunctionOutput(output_info={"rule": "no SSNs"}, tripwire_triggered="ssn" in str(user_input).lower())


ok_json = WorkerResult(topic="visual_art", status="success", failure_type=None, retryable=False,
                       attempted_query="AI art generation", findings=[Finding(title="AI art report", source="archive#12")],
                       alternative_approaches=[], coverage_impact=None).model_dump_json()

WORKERS = {
    "success": (Agent(name="Art", model=ScriptedModel([[assistant_message(ok_json)]]), output_type=WorkerResult),
                "Research AI in visual art"),
    "max_turns": (Agent(name="Music", model=looping_model(8), tools=[search_notes], output_type=WorkerResult),
                  "Research AI in music"),
    "guardrail": (Agent(name="Lit", model=ScriptedModel([[assistant_message("{}")]]), output_type=WorkerResult,
                        input_guardrails=[no_personal_data]), "Research the author with SSN 123-45-6789"),
    "billing 429": (Agent(name="Lit", model=ScriptedModel([ModelStep.raise_error(billing_429())]),
                          output_type=WorkerResult), "Research AI in literature"),
    "revoked key": (Agent(name="Lit", model=ScriptedModel([ModelStep.raise_error(
                        raise_from(lambda r: error_response(401, "invalid_api_key", "invalid_request_error")))]),
                          output_type=WorkerResult), "Research AI in literature"),
    "model timeout": (Agent(name="Lit", model=ScriptedModel([ModelStep.respond(slow_responder)]), output_type=WorkerResult,
                            model_settings=ModelSettings(timeout=0.1)), "Research AI in literature"),
    "refusal": (Agent(name="Lit", model=ScriptedModel([[refusal]]), output_type=WorkerResult), "Research AI in literature"),
}
for label, (agent, task) in WORKERS.items():
    r = await run_worker(agent, label, task, run_config=OFFLINE)
    print(f"{label:<14} status={r.status:<16} failure_type={str(r.failure_type):<15} retryable={r.retryable!s:<6}"
          f"findings={len(r.findings)}  impact={r.coverage_impact}")

> **Observe:** six very different failures, one shape. The `max_turns` worker still delivered the findings its tools had returned before the limit. The revoked key (`401`) becomes `failure_type="permission"`, which is the value the coordinator's escalation rule in 10.7 keys on. `ModelTimeoutError` and `ToolTimeoutError` are `AgentsException`s, **not** Python's `TimeoutError`, so an `except TimeoutError` around `Runner.run` would let them escape; `run_worker` names them explicitly.

**Code orchestration: `gather` versus `TaskGroup`.** Three fake workers; the third raises after a short delay.

In [ ]:
async def fake_worker(name: str, delay: float, fail: bool, log: list) -> str:
    try:
        await asyncio.sleep(delay)
        if fail:
            raise TimeoutError(f"{name} timed out")
        log.append(f"{name}: finished")
        return f"{name} result"
    except asyncio.CancelledError:
        log.append(f"{name}: CANCELLED")
        raise


log: list[str] = []
try:
    async with asyncio.TaskGroup() as tg:
        for name, delay, fail in [("visual_art", 0.10, False), ("music", 0.10, False), ("literature", 0.02, True)]:
            tg.create_task(fake_worker(name, delay, fail, log))
except* TimeoutError as group:
    log.append(f"TaskGroup raised: {group.exceptions[0]}")
print("TaskGroup     :", log)

log = []
results = await asyncio.gather(*(fake_worker(n, d, f, log) for n, d, f in
                                 [("visual_art", 0.10, False), ("music", 0.10, False), ("literature", 0.02, True)]),
                               return_exceptions=True)
print("gather(return_exceptions=True):", [r if isinstance(r, str) else f"{type(r).__name__}: {r}" for r in results])

> **Observe:** with `TaskGroup`, the two healthy workers were **cancelled** because the third failed. `gather(return_exceptions=True)`, or a per-worker wrapper like `run_worker`, keeps their results.

**Agents as tools.** The manager sees whatever the nested run produces as a tool output. Make that explicit JSON with `custom_output_extractor`, and replace `as_tool`'s generic default `failure_error_function`.

In [ ]:
async def as_json(nested: RunResult) -> str:            # custom_output_extractor must be async
    return nested.final_output.model_dump_json()


async def manager_sees(worker_model: ScriptedModel, **as_tool_kwargs) -> str:
    worker = Agent(name="Music researcher", model=worker_model, output_type=WorkerResult)
    tool = worker.as_tool(tool_name="research_music", tool_description="Research AI impact on music. Returns WorkerResult JSON.",
                          custom_output_extractor=as_json, max_turns=6,
                          run_config=OFFLINE, **as_tool_kwargs)      # run_config: offline demo, nothing exported
    manager_model = ScriptedModel([[function_call("research_music", {"input": "AI in music"}, call_id="m1")],
                                   [assistant_message("Report written.")]])
    result = await Runner.run(Agent(name="Coordinator", model=manager_model, tools=[tool]), "Write the report",
                              run_config=OFFLINE)
    return next(i.output for i in result.new_items if i.type == "tool_call_output_item")


music_ok = WorkerResult(topic="music", status="success", failure_type=None, retryable=False,
                        attempted_query="AI music generation", alternative_approaches=[], coverage_impact=None,
                        findings=[Finding(title="AI songwriting tools survey", source="archive#7")]).model_dump_json()
print("worker succeeds              :", await manager_sees(ScriptedModel([[assistant_message(music_ok)]])))
print("worker raises, default       :", await manager_sees(ScriptedModel([ModelStep.raise_error(overloaded_503())])))
print("worker raises, tool_error_as_data:",
      await manager_sees(ScriptedModel([ModelStep.raise_error(overloaded_503())]), failure_error_function=tool_error_as_data))

> **Try it:** `tool_error_as_data` maps the nested `InternalServerError` to `unknown`. Add a branch for `openai.APIError` that uses `classify_openai_error`, so the manager learns that a `503` is transient.

## 10.6 Local recovery before propagation

Recover where the knowledge is: the tool retries a quick idempotent read once, the worker retries a transient tool error once with a changed query, and only the coordinator decides whether a section can be missing. Every layer has a small budget **and** a deadline. [Theory](README.md#106-local-recovery-before-propagation)

In [ ]:
async def retry_transient(op, *, attempts: int = 2, deadline_s: float = 30.0, base_delay: float = 0.5):
    """Retry TimeoutError/ConnectionError. Every attempt AND every backoff stays inside deadline_s."""
    loop = asyncio.get_running_loop()
    deadline = loop.time() + deadline_s
    for attempt in range(1, attempts + 1):
        try:
            async with asyncio.timeout_at(deadline):    # an attempt can't outlive the budget (raises TimeoutError)
                return await op()
        except (TimeoutError, ConnectionError):
            delay = min(base_delay * 2 ** attempt, 4.0) * (0.75 + random.random() / 4)
            if attempt == attempts or loop.time() + delay >= deadline:
                raise                                   # let failure_error_function categorize it
            await asyncio.sleep(delay)


def flaky(failures: list[Exception]):
    """An async operation that raises the given exceptions in order, then succeeds. Counts calls."""
    state = {"calls": 0}

    async def op():
        state["calls"] += 1
        if failures:
            raise failures.pop(0)
        return "data"
    return op, state


for label, failures in {"one transient blip": [TimeoutError()],
                        "persistent timeout": [TimeoutError()] * 5,
                        "permission error": [PermissionError("403")]}.items():
    op, state = flaky(list(failures))
    try:
        outcome = await retry_transient(op, attempts=2, base_delay=0.01)
    except Exception as exc:
        outcome = f"raised {type(exc).__name__}"
    print(f"{label:<20} -> {outcome:<26} calls={state['calls']}")

# The deadline bounds the whole budget: a hung call is cut off, and a backoff that would cross it is skipped.
calls = {"n": 0}


async def hangs():
    calls["n"] += 1
    await asyncio.sleep(5)

start = time.monotonic()
try:
    await retry_transient(hangs, attempts=3, deadline_s=0.1, base_delay=0.01)
except TimeoutError:
    print(f"{'hung upstream':<20} -> raised TimeoutError after {time.monotonic() - start:.2f}s  calls={calls['n']}")

During a sustained outage, a **circuit breaker** stops every worker from spending its retry budget on a dependency that is known to be down.

In [ ]:
class CircuitBreaker:
    """closed -> OPEN after `threshold` consecutive failures -> HALF-OPEN after `cooldown_s`, which admits exactly
    ONE probe; the probe's outcome closes the breaker or re-opens it for another cooldown."""

    def __init__(self, threshold: int = 3, cooldown_s: float = 30.0, clock=time.monotonic):
        self.threshold, self.cooldown_s, self.clock = threshold, cooldown_s, clock
        self.failures, self.opened_at, self.probe_in_flight = 0, None, False

    @property
    def state(self) -> str:
        if self.opened_at is None:
            return "closed"
        return "half_open" if self.clock() - self.opened_at >= self.cooldown_s else "open"

    def allow(self) -> bool:
        state = self.state
        if state == "closed":
            return True
        if state == "half_open" and not self.probe_in_flight:
            self.probe_in_flight = True                 # one probe; everyone else keeps failing fast
            return True
        return False

    def record(self, ok: bool) -> None:
        if self.probe_in_flight:                        # the probe decides
            self.probe_in_flight = False
            self.failures, self.opened_at = (0, None) if ok else (self.threshold, self.clock())
            return
        self.failures = 0 if ok else self.failures + 1
        if self.failures >= self.threshold:
            self.opened_at = self.clock()


now = [0.0]
breaker = CircuitBreaker(threshold=3, cooldown_s=30, clock=lambda: now[0])
for t, ok in [(0, False), (1, False), (2, False), (3, None), (10, None), (33, False), (40, None), (64, True), (65, True)]:
    now[0] = t
    if not breaker.allow():
        print(f"t={t:>2}s  breaker {breaker.state.upper()} -> fail fast, no request sent")
        continue
    breaker.record(ok)
    print(f"t={t:>2}s  request sent, ok={ok}, breaker now {breaker.state}")

# Half-open admits ONE probe, even if many workers ask before it finishes.
now[0] = 0.0
burst = CircuitBreaker(threshold=1, cooldown_s=30, clock=lambda: now[0])
burst.record(False)
now[0] = 31
print("three workers right after the cooldown:", [burst.allow() for _ in range(3)])

A per-worker deadline belongs to the coordinator, so one slow worker cannot hold the whole report hostage:

In [ ]:
async def slow_worker():
    await asyncio.sleep(5)
    return "late result"


try:
    async with asyncio.timeout(0.1):
        await slow_worker()
except TimeoutError:
    print("worker exceeded its 0.1 s deadline -> coordinator records failure_type='timeout' and moves on")

> **Try it:** give `retry_transient` a `breaker: CircuitBreaker | None` argument: check `breaker.allow()` before each attempt and `breaker.record(...)` after it.

## 10.7 Coordinator decisions and the anti-patterns

Given the `WorkerResult`s, the coordinator decides per topic: use, re-dispatch once with a changed approach, accept partial, escalate, or stop everything. Make the rules explicit in code. [Theory](README.md#107-coordinator-decisions-and-the-anti-patterns)

In [ ]:
Decision = Literal["use", "use_empty", "redispatch", "accept_partial", "escalate_access",
                   "route_to_review", "stop_workflow", "report_gap"]


def decide(r: WorkerResult, *, already_redispatched: bool = False) -> Decision:
    if r.status == "success":
        return "use" if r.findings else "use_empty"
    if r.failure_type == "permission":
        return "escalate_access"
    if r.failure_type == "policy_blocked":
        return "route_to_review"
    if r.failure_type == "rate_limited" and not r.retryable:
        return "stop_workflow"                         # billing / quota: no new work for anyone
    if r.retryable and r.alternative_approaches and not already_redispatched:
        return "redispatch"
    return "accept_partial" if r.findings else "report_gap"


def worker_result(topic, status, failure_type=None, retryable=False, findings=0, alternatives=()):
    return WorkerResult(topic=topic, status=status, failure_type=failure_type, retryable=retryable,
                        attempted_query=f"AI in {topic}", findings=[Finding(title=f"{topic} {i}", source="archive")
                                                                    for i in range(findings)],
                        alternative_approaches=list(alternatives), coverage_impact=None if status == "success" else "gap")


SAMPLES_7 = [
    worker_result("visual_art", "success", findings=2),
    worker_result("literature", "success"),
    worker_result("music", "partial_failure", "timeout", True, 1, ["narrower query"]),
    worker_result("film", "failure", "permission"),
    worker_result("games", "failure", "policy_blocked"),
    worker_result("design", "failure", "rate_limited", False),
    worker_result("theatre", "failure", "timeout", True, 0, []),
]
for r in SAMPLES_7:
    print(f"{r.topic:<11} {r.status:<16} {str(r.failure_type):<15} retryable={r.retryable!s:<6} -> {decide(r)}")
print("music after one re-dispatch ->", decide(SAMPLES_7[2], already_redispatched=True))


def too_many_gaps(results: list[WorkerResult], max_gap_ratio: float = 0.5) -> bool:
    gaps = sum(r.status != "success" for r in results)
    return gaps / len(results) > max_gap_ratio


print("\nmore than half the sections have gaps -> mark the report partial / ask the user:", too_many_gaps(SAMPLES_7))

> **Observe:** only one row re-dispatches, and only once. Permission, policy and billing failures never retry; the empty literature search is a **use**, not a failure.
>
> **Try it:** add a rule: if any result is `stop_workflow`, every other `redispatch` becomes `report_gap` (no new model calls while the wallet is empty).

## 10.8 Coverage annotations in the final synthesis

Compute the coverage map **in code** from worker statuses and give it to the synthesis step as data. If a model writes the prose, check its coverage claims against the map before publishing. [Theory](README.md#108-coverage-annotations-in-the-final-synthesis)

In [ ]:
LABEL = {"success": "FULL COVERAGE", "partial_failure": "PARTIAL COVERAGE", "failure": "NOT COVERED"}


def render_report(title: str, results: list[WorkerResult]) -> str:
    lines = [f"## {title}", ""]
    for r in results:
        label = LABEL[r.status] + (f": {r.failure_type}" if r.failure_type else "")
        if r.status == "success" and not r.findings:
            label += "; searched, no matching records"
        lines.append(f"### {r.topic.replace('_', ' ').title()} ({label})")
        lines += [f"- {f.title} ({f.source})" for f in r.findings] or ["- No findings."]
        if r.coverage_impact:
            lines.append(f"> Coverage note: {r.coverage_impact}")
        lines.append("")
    gaps = [r.topic for r in results if r.status != "success"]
    lines.append(f"**Coverage:** {len(results) - len(gaps)}/{len(results)} sections complete"
                 + (f"; gaps: {', '.join(gaps)}" if gaps else ""))
    return "\n".join(lines)


SAMPLE_RESULTS = [
    WorkerResult(topic="film", status="success", failure_type=None, retryable=False,
                 attempted_query="AI storyboarding in film studios",
                 findings=[Finding(title="Studio survey on AI storyboarding", source="archive#21")],
                 alternative_approaches=[], coverage_impact=None),
    WorkerResult(topic="writing", status="partial_failure", failure_type="timeout", retryable=True,
                 attempted_query="AI drafting tools at publishers; AI translation of novels",
                 findings=[Finding(title="Publisher survey on AI drafting tools", source="archive#9")],
                 alternative_approaches=["Query the translators' association reports instead"],
                 coverage_impact="Translation search timed out twice; translation is missing."),
    WorkerResult(topic="music", status="success", failure_type=None, retryable=False,
                 attempted_query="AI-composed chart singles", findings=[], alternative_approaches=[],
                 coverage_impact=None),
]
print(render_report("AI impact on creative industries", SAMPLE_RESULTS))

When a model writes the final report, make its coverage claims typed data and verify them with an **output guardrail** that compares them to the code-computed map (passed in the run context). Here the scripted "writer" claims full coverage for writing:

In [ ]:
class SectionClaim(BaseModel):
    topic: str
    coverage: Literal["FULL", "PARTIAL", "NOT_COVERED"]
    summary: str


class SynthesisReport(BaseModel):
    sections: list[SectionClaim]


CLAIM_FOR = {"success": "FULL", "partial_failure": "PARTIAL", "failure": "NOT_COVERED"}


@output_guardrail
def coverage_matches(ctx: RunContextWrapper, agent: Agent, output: SynthesisReport) -> GuardrailFunctionOutput:
    expected = ctx.context["coverage"]                       # computed in code from WorkerResults
    wrong = {s.topic: (s.coverage, expected.get(s.topic)) for s in output.sections
             if expected.get(s.topic) != s.coverage}
    missing = sorted(set(expected) - {s.topic for s in output.sections})
    return GuardrailFunctionOutput(output_info={"wrong": wrong, "missing": missing},
                                   tripwire_triggered=bool(wrong or missing))


coverage = {r.topic: CLAIM_FOR[r.status] for r in SAMPLE_RESULTS}
overclaiming = SynthesisReport(sections=[SectionClaim(topic=t, coverage="FULL", summary="...") for t in coverage])
honest = SynthesisReport(sections=[SectionClaim(topic=t, coverage=c, summary="...") for t, c in coverage.items()])

for label, draft in {"over-claiming draft": overclaiming, "honest draft": honest}.items():
    writer = Agent(name="Writer", model=ScriptedModel([[assistant_message(draft.model_dump_json())]]),
                   output_type=SynthesisReport, output_guardrails=[coverage_matches])
    try:
        await Runner.run(writer, "Write the report", context={"coverage": coverage}, run_config=OFFLINE)
        print(f"{label:<20} -> published")
    except OutputGuardrailTripwireTriggered as exc:
        print(f"{label:<20} -> BLOCKED: {exc.guardrail_result.output.output_info}")

> **Observe:** the guardrail is deterministic and cheap: no second model call, no judgment. The model writes prose; code owns the facts about coverage.

## 10.9 Tracing for debugging failures

Handled tool errors don't fail a run, but the Agents SDK records them on the **function span**. Add a `custom_span` per worker and call `set_error` when it returns a partial result, and group everything in one `trace(...)` with a `group_id`. Here a local processor captures spans instead of exporting them. [Theory](README.md#109-tracing-for-debugging-failures)

In [ ]:
class SpanCollector(TracingProcessor):
    """Keeps finished spans in memory (a local stand-in for the Traces dashboard)."""

    def __init__(self):
        self.spans = []

    def on_trace_start(self, trace): pass
    def on_trace_end(self, trace): pass
    def on_span_start(self, span): pass
    def on_span_end(self, span): self.spans.append(span)
    def shutdown(self): pass
    def force_flush(self): pass


@contextmanager
def capture_spans():
    """Route traces to a SpanCollector for the duration of the block, then restore the default exporter."""
    collector = SpanCollector()
    set_trace_processors([collector])
    try:
        yield collector
    finally:
        set_trace_processors([default_processor()])


def span_summary(collector: SpanCollector) -> None:
    for span in collector.spans:
        data = span.span_data
        name = getattr(data, "name", None) or ""
        flag = f"  ERROR {span.error}" if span.error else ""
        print(f"  {data.type:<9} {name:<22}{flag}")


failing_tool = make_archive_tool(failure_error_function=tool_error_as_data)
with capture_spans() as collector:
    with trace("Creative industries report", group_id="report-2026-10-03"):
        with custom_span("worker:music") as span:
            model = ScriptedModel([[function_call("search_archive", ARGS, call_id="t1")], [assistant_message("partial")]])
            await Runner.run(Agent(name="Music researcher", model=model, tools=[failing_tool]), "Research AI music")
            span.set_error(SpanError(message="partial_failure", data={"failure_type": "timeout", "retryable": True}))
span_summary(collector)

> **Observe:** the run "succeeded", yet the function span carries `Error running tool (non-fatal)` with the tool name and the original error text (including the internal hostname, which is why traces are access-controlled data). That raw text appears only because `failing_tool` uses a **custom** formatter: with the SDK's default formatter, 0.23.1 stores `Tool execution failed. Error details are redacted.` instead. Your `custom_span` makes the worker's partial outcome visible at a glance.

With `RunConfig(trace_include_sensitive_data=False)`, spans keep their structure but drop inputs, outputs and error details. Compare:

In [ ]:
with capture_spans() as collector:
    model = ScriptedModel([[function_call("search_archive", ARGS, call_id="t1")], [assistant_message("partial")]])
    await Runner.run(Agent(name="Music researcher", model=model, tools=[failing_tool]), "Research AI music",
                     run_config=RunConfig(trace_include_sensitive_data=False))
fn = next(s for s in collector.spans if s.span_data.type == "function")
print("function span error :", fn.error)
print("function span input :", fn.span_data.input, "| output:", fn.span_data.output)

> **Try it:** inside the first trace, also log `exc.request_id` from `overloaded_503()` into the custom span's `data`. In production, those IDs are what support asks for.

## 10.10 Background mode and managed-runtime failure statuses

Long-running work doesn't raise in your process; it ends in a **terminal status** you must read. A background response ends `completed`, `failed`, `cancelled` or `incomplete`; an Agents API turn can end `failed` with a stable `error.code`. Classify before you retry, and never replay work that already had side effects. [Theory](README.md#1010-background-mode-and-managed-runtime-failure-statuses)

In [ ]:
def background_server(terminal: dict) -> ScriptedServer:
    """Mock API: POST /responses returns `queued`; each GET returns `in_progress` twice, then `terminal`."""
    polls = {"n": 0}

    def handler(request):
        base = {"id": "resp_bg_1", "object": "response", "created_at": 0, "model": MODEL, "output": [],
                "parallel_tool_calls": True, "tool_choice": "auto", "tools": [], "background": True}
        if request.method == "POST":
            return httpx2.Response(200, json={**base, "status": "queued"})
        polls["n"] += 1
        return httpx2.Response(200, json={**base, **({"status": "in_progress"} if polls["n"] <= 2 else terminal)})
    return ScriptedServer(handler)


def poll_until_terminal(llm: OpenAI, response_id: str, *, interval_s: float = 2.0, max_wait_s: float = 600):
    start = time.monotonic()
    resp = llm.responses.retrieve(response_id)
    while resp.status in {"queued", "in_progress"}:
        if time.monotonic() - start > max_wait_s:
            raise TimeoutError(f"{response_id} still {resp.status} after {max_wait_s}s; cancel or keep waiting")
        time.sleep(interval_s)
        resp = llm.responses.retrieve(response_id)
    return resp


RETRYABLE_RESPONSE_CODES = {"server_error", "rate_limit_exceeded", "vector_store_timeout"}


# `incomplete` has several causes; dispatch on incomplete_details.reason. Partial output is never trusted.
INCOMPLETE_ACTION = {
    "max_output_tokens": "budget: raise max_output_tokens, lower reasoning effort or split the task",
    "content_filter": "policy: output was filtered; don't resubmit unchanged, route to review",
    "steered": "steered over WebSocket: follow the successor response, don't resubmit",
}


def handle_terminal(resp) -> str:
    match resp.status:
        case "completed":
            return "use output_text (still check for refusals)"
        case "incomplete":
            reason = resp.incomplete_details.reason if resp.incomplete_details else None
            action = INCOMPLETE_ACTION.get(reason, f"unrecognized reason ({reason}): log it, no automatic retry")
            return f"incomplete/{reason} -> {action}; never parse partial JSON"
        case "failed" if resp.error.code in RETRYABLE_RESPONSE_CODES:
            return f"transient ({resp.error.code}): bounded retry, check side effects first"
        case "failed":
            return f"non-retryable ({resp.error.code}): fix the request or escalate"
        case "cancelled":
            return "intentional: do not resubmit automatically"
    return f"unexpected status {resp.status}"


TERMINALS = {
    "completed": {"status": "completed", "output": [{"type": "message", "id": "m", "role": "assistant", "status": "completed",
                                                     "content": [{"type": "output_text", "text": "Done.", "annotations": []}]}]},
    "incomplete: tokens": {"status": "incomplete", "incomplete_details": {"reason": "max_output_tokens"}},
    "incomplete: filter": {"status": "incomplete", "incomplete_details": {"reason": "content_filter"}},
    "failed: server_error": {"status": "failed", "error": {"code": "server_error", "message": "The server had an error."}},
    "failed: misalignment": {"status": "failed", "error": {"code": "misalignment_policy_violation", "message": "Blocked."}},
    "cancelled": {"status": "cancelled"},
}
for label, terminal in TERMINALS.items():
    server = background_server(terminal)
    llm = mock_client(server)
    created = llm.responses.create(model=MODEL, input="Long analysis ...", background=True, store=True)
    final = poll_until_terminal(llm, created.id, interval_s=0.01)
    print(f"{label:<22} requests={server.requests}  -> {handle_terminal(final)}")

> **Observe:** `incomplete` is not `failed`, and not every `incomplete` is a budget problem. With `max_output_tokens`, a resubmission with the same settings hits the same wall; with `content_filter`, generation was stopped by filtering, so it belongs with policy, not with settings. The SDK's reason enum (`openai` 3.24) also lists `steered` and `max_messages`; anything you don't recognize gets logged, not retried. `misalignment_policy_violation` is a `failed` response that must never be retried.

Agents API turns use the SDK's `Turn` type with a `SessionTurnError`. The mapping below is **our** recovery plan built from the error-and-recovery guide's procedure, not an official table.

> **Version note (checked 2026-10-04):** the guide lists `project_spend_limit_exceeded`, `organization_spend_limit_exceeded` and `organization_usage_limit_exceeded` as turn codes, but the `SessionTurnError.code` enum in `openai` 3.24 doesn't include them yet. `Turn.model_validate` rejects them; the client's default (non-strict) response parsing keeps them. So the demo builds turns the way the client does (`construct_type`), and your code should compare codes as strings and log unknown ones, never rely on the enum being complete.

In [ ]:
from openai._models import construct_type          # how the client parses API responses (non-strict)
from openai.types.beta.agents.sessions import Turn

TURN_PLAN = {
    "transient": ({"server_overloaded", "rate_limit_exceeded", "server_error", "connection_failed",
                   "request_timeout", "internal_error", "flex_unavailable"},
                  "1) retrieve session, turn, saved items  2) check completed actions  "
                  "3) wait (Retry-After, attempt limit)  4) when idle, ask to continue ONLY unfinished work"),
    "business": ({"credit_balance_exhausted", "project_spend_limit_exceeded", "organization_spend_limit_exceeded",
                  "organization_usage_limit_exceeded", "usage_limit_exceeded", "session_budget_exceeded"},
                 "fix the credits or limit first (session budget: start a new session); alert the owner; "
                 "no automatic retry"),
    "validation": ({"invalid_request", "resource_not_found", "context_length_exceeded"},
                   "fix the input (shorter context, valid IDs) before any retry"),
    "permission": ({"authentication_error"}, "fix credentials; escalate"),
    "policy": ({"misalignment_policy_violation", "cyber_policy"}, "stop dispatching actions for this conversation; review"),
}


def turn_recovery(turn: Turn) -> str:
    if turn.status != "failed":
        return f"status {turn.status}: nothing to recover"
    for category, (codes, plan) in TURN_PLAN.items():
        if turn.error.code in codes:
            return f"{category}: {plan}"
    return f"other ({turn.error.code}): inspect the session; it may need a new session"


for code in ["server_overloaded", "project_spend_limit_exceeded", "context_length_exceeded", "sandbox_error"]:
    turn = construct_type(type_=Turn, value={"id": "turn_1", "agent_id": "agent_1", "session_id": "sess_1",
                                             "created_at": 0, "object": "agent.session.turn", "status": "failed",
                                             "error": {"code": code, "message": "..."}})
    print(f"{code:<30} -> {turn_recovery(turn)}")

In [ ]:
# (live) ~ $0.002. A real background response: poll it to a terminal status, then classify it.
if not os.getenv("OPENAI_API_KEY"):
    print("Skipping (live): set OPENAI_API_KEY in .env")
else:
    bg = client.responses.create(model=MODEL, input="List three risks of retrying side-effecting tool calls. One line each.",
                                 background=True, store=True, max_output_tokens=600)
    print("created:", bg.id, bg.status)
    final = poll_until_terminal(client, bg.id, interval_s=2, max_wait_s=120)
    print("terminal:", final.status, "->", handle_terminal(final))
    if final.status == "completed":
        print(final.output_text)

> **Try it:** start a second background response and call `client.responses.cancel(bg.id)` right away. Poll it and check what `handle_terminal` says. Cancelling twice is idempotent.

## Mini-project: a fault-tolerant research coordinator

Three worker agents (visual art, music, literature) research "AI impact on creative industries". Faults are injected at every layer:

| Fault | Where | Expected handling |
|---|---|---|
| `503 server_is_overloaded` on the visual-art worker's first model call | model call | `ModelRetrySettings` retries once → recovered |
| First music archive search times out once | tool | `retry_transient` inside the tool → recovered |
| "AI vocal synthesis" times out every time | tool | tool reports `transient`; worker retries once with a narrower query; coordinator re-dispatches once; still failing → **partial** |
| Licensed gallery-sales database answers `403 subscription expired` | tool | `permission`, not retryable → partial, **escalate access** |
| Literature search runs and finds nothing | tool | **valid empty**: success, "searched, no matching records" |

Offline, each worker's "LLM" is a `ScriptedModel` driven by a small deterministic policy that follows the worker instructions literally. The last cell runs the same workflow live on `OPENAI_MODEL`.

In [ ]:
DATA = {
    ("visual_art", "AI art generation"): [{"title": "AI art report", "source": "archive#12"}],
    ("music", "AI music generation"): [{"title": "AI songwriting tools survey", "source": "archive#7"}],
    ("music", "AI singing-voice tools in trade press"): [{"title": "Trade-press review of AI singing voices",
                                                         "source": "press#4"}],   # used by Try it (1)
}


class InjectedArchive:
    """Fake backends with injected faults. Counts every backend call."""

    def __init__(self):
        self.calls: Counter = Counter()
        self.blip_pending = {"music"}                        # first music search times out once

    async def search(self, query: str, topic: str) -> list[dict]:
        self.calls[f"archive:{topic}"] += 1
        await asyncio.sleep(0)
        if topic == "music" and "vocal" in query.lower():
            raise TimeoutError("shard music-vocals not answering")          # persistent
        if topic in self.blip_pending:
            self.blip_pending.discard(topic)
            raise TimeoutError("transient blip")                             # recovers on retry
        return DATA.get((topic, query), [])

    async def licensed(self, query: str, topic: str) -> list[dict]:
        self.calls[f"licensed:{topic}"] += 1
        if topic == "visual_art":
            raise ToolFailure("permission", "403: subscription to the gallery-sales database expired.", False,
                              "Do not retry. Report the source as not covered and escalate access.")
        return []


ARCHIVE = InjectedArchive()


@function_tool(failure_error_function=tool_error_as_data, timeout=10)
async def search_archive(query: str, topic: str) -> str:
    """Search the research archive. Returns JSON with status ok (results may be empty) or status error.

    Args:
        query: Search terms.
        topic: One of visual_art, music, literature.
    """
    hits = await retry_transient(lambda: ARCHIVE.search(query, topic), attempts=2, deadline_s=5, base_delay=0.01)
    return json.dumps({"status": "ok", "results": hits, "note": None if hits else "searched, 0 matches"})


@function_tool(failure_error_function=tool_error_as_data, timeout=10)
async def search_licensed_db(query: str, topic: str) -> str:
    """Search the licensed market-data database.

    Args:
        query: Search terms.
        topic: One of visual_art, music, literature.
    """
    hits = await ARCHIVE.licensed(query, topic)
    return json.dumps({"status": "ok", "results": hits, "note": None if hits else "searched, 0 matches"})


WORKER_INSTRUCTIONS = (
    "Research one topic with the search tools. Every tool returns JSON with a status. If a search fails, read its "
    "category: retry a transient error at most once with a narrower query; never retry validation, permission or "
    "business errors. Return a WorkerResult. status=success only if every planned search ran (an empty result counts "
    "as run). Keep every finding you got, even when status is partial_failure. Put each failed query in "
    "coverage_impact as 'Not covered: <query>' and suggest alternative_approaches for transient failures."
)

In [ ]:
FAILURE_TYPE = {"transient": "timeout", "permission": "permission", "validation": "validation"}


class ScriptedResearcher:
    """Deterministic stand-in for the worker LLM: follows WORKER_INSTRUCTIONS literally."""

    def __init__(self, topic: str, plan: list[tuple[str, str, str | None]]):
        self.topic = topic
        self.pending = [(tool, query, narrower, query) for tool, query, narrower in plan]
        self.current: tuple | None = None
        self.log: list[tuple[str, str, dict]] = []        # (planned query, query actually sent, parsed tool output)
        self.n = 0

    def respond(self, call):
        last = call.input[-1] if isinstance(call.input, list) and call.input else None
        if isinstance(last, dict) and last.get("type") == "function_call_output" and self.current:
            tool, query, narrower, origin = self.current
            try:
                out = json.loads(last["output"])
            except json.JSONDecodeError:                   # e.g. a plain-text tool timeout message
                out = {"status": "error", "category": "transient", "retryable": True, "message": last["output"]}
            self.log.append((origin, query, out))
            if out.get("status") == "error" and out.get("category") == "transient" and narrower:
                self.pending.insert(0, (tool, narrower, None, origin))     # one retry, changed query
            self.current = None
        if self.pending:
            self.current = self.pending.pop(0)
            self.n += 1
            tool, query, _, _ = self.current
            return [function_call(tool, {"query": query, "topic": self.topic}, call_id=f"{self.topic}_{self.n}")]
        return [assistant_message(self.result().model_dump_json())]

    def result(self) -> WorkerResult:
        last_attempt = {origin: out for origin, _, out in self.log}       # the final outcome per planned search
        failed = {origin: out for origin, out in last_attempt.items() if out.get("status") == "error"}
        covered = [origin for origin, out in last_attempt.items() if out.get("status") == "ok"]
        return WorkerResult(
            topic=self.topic,
            status="success" if not failed else ("partial_failure" if covered else "failure"),
            failure_type=most_actionable(FAILURE_TYPE.get(out.get("category"), "unknown") for out in failed.values()),
            retryable=bool(failed) and all(out.get("retryable") for out in failed.values()),
            attempted_query="; ".join(query for _, query, _ in self.log),
            findings=[Finding(**hit) for _, _, out in self.log if out.get("status") == "ok" for hit in out["results"]],
            alternative_approaches=["Search the trade-press source for this sub-topic"]
            if any(out.get("category") == "transient" for out in failed.values()) else [],
            coverage_impact="; ".join(f"Not covered: {origin} ({out.get('category')})"
                                      for origin, out in failed.items()) or None,
        )

    def model(self, fail_first_call: Exception | None = None) -> ScriptedModel:
        steps = [ModelStep.raise_error(fail_first_call)] if fail_first_call else []
        return ScriptedModel(steps + [ModelStep.respond(self.respond) for _ in range(12)])


PLANS = {   # what each scripted worker "decides" to search: (tool, query, narrower retry query)
    "visual_art": [("search_archive", "AI art generation", None), ("search_licensed_db", "AI art market sales", None)],
    "music": [("search_archive", "AI music generation", None),
              ("search_archive", "AI vocal synthesis", "AI vocal synthesis plugins")],
    "literature": [("search_archive", "AI-written novels", None)],
}
REDISPATCH_PLANS = {"music": [("search_archive", "AI vocal tools in trade press", None)]}
TASKS = {"visual_art": "Research AI impact on visual art.", "music": "Research AI impact on music.",
         "literature": "Research AI impact on literature."}
RETRY = ModelRetrySettings(max_retries=2, backoff=FAST_BACKOFF, policy=SAFE_POLICY)


def make_worker(topic: str, *, live: bool, redispatch: bool = False) -> Agent:
    if live:
        model = MODEL
    else:
        plan = REDISPATCH_PLANS[topic] if redispatch else PLANS[topic]
        fault = overloaded_503() if topic == "visual_art" and not redispatch else None
        model = ScriptedResearcher(topic, plan).model(fail_first_call=fault)
    return Agent(name=f"{topic} researcher", instructions=WORKER_INSTRUCTIONS, model=model,
                 tools=[search_archive, search_licensed_db], output_type=WorkerResult,
                 model_settings=ModelSettings(retry=RETRY))

In [ ]:
def merge(first: WorkerResult, second: WorkerResult) -> WorkerResult:
    """Combine a re-dispatch with the original attempt. Contract: the re-dispatch task covers exactly the
    first attempt's gaps, so its own status says whether those gaps are now covered. Every finding is kept,
    every failure stays visible, and the status, failure type and retryable flag are recomputed from both."""
    findings = first.findings + [f for f in second.findings if f not in first.findings]
    history = f"{first.coverage_impact} | re-dispatched once ({second.attempted_query})"
    if second.status == "success":
        return first.model_copy(update={"status": "success", "failure_type": None, "retryable": False,
                                        "findings": findings, "alternative_approaches": [],
                                        "coverage_impact": f"{history}: gaps covered."})
    return first.model_copy(update={
        "status": "partial_failure" if findings else "failure",
        "failure_type": most_actionable([first.failure_type, second.failure_type]),   # a new 403 or 429 wins
        "retryable": first.retryable and second.retryable,
        "findings": findings,
        "alternative_approaches": second.alternative_approaches,
        "coverage_impact": f"{history}: still not covered. {second.coverage_impact or ''}".strip(),
    })


async def coordinate(*, live: bool, per_worker_deadline_s: float = 90) -> list[WorkerResult]:
    async def guarded(topic: str, redispatch: bool = False, task: str | None = None) -> WorkerResult:
        task = task or TASKS[topic]
        with custom_span(f"worker:{topic}" + (":redispatch" if redispatch else "")) as span:
            try:
                async with asyncio.timeout(per_worker_deadline_s):
                    r = await run_worker(make_worker(topic, live=live, redispatch=redispatch), topic, task)
            except TimeoutError:                     # asyncio.timeout's builtin TimeoutError (the worker deadline)
                r = WorkerResult(topic=topic, status="failure", failure_type="timeout", retryable=True,
                                 attempted_query=task, findings=[], alternative_approaches=[],
                                 coverage_impact=f"Worker exceeded its {per_worker_deadline_s}s deadline.")
            except Exception as exc:
                # Residual failures (UserError, a bug in our code) still become a WorkerResult at this boundary.
                # asyncio.CancelledError is a BaseException, so cancellation still propagates.
                r = WorkerResult(topic=topic, status="failure", failure_type="unknown", retryable=False,
                                 attempted_query=task, findings=[], alternative_approaches=[],
                                 coverage_impact=f"Worker crashed ({type(exc).__name__}); see logs.")
            if r.status != "success":
                span.set_error(SpanError(message=r.status, data={"failure_type": r.failure_type,
                                                                 "retryable": r.retryable}))
            return r

    results = list(await asyncio.gather(*(guarded(t) for t in TASKS)))     # guarded() always returns a WorkerResult
    decisions = [decide(r) for r in results]
    for r, decision in zip(results, decisions):
        print(f"  decision {r.topic:<11} -> {decision}")
        if decision == "escalate_access":
            print(f"  ESCALATION: {r.topic}: {r.coverage_impact}")
    if "stop_workflow" in decisions:                 # workflow-wide stop: checked BEFORE any new work starts
        print("  billing/quota failure: no re-dispatch, no further model calls in this workflow")
        return results
    for i, (r, decision) in enumerate(zip(results, decisions)):
        if decision != "redispatch":
            continue
        task = f"{TASKS[r.topic]} Previous attempt: {r.coverage_impact}. Try instead: {r.alternative_approaches[0]}."
        second = await guarded(r.topic, redispatch=True, task=task)
        results[i] = merge(r, second)
        final = decide(results[i], already_redispatched=True)
        print(f"  re-dispatch {r.topic:<8} -> {second.status}; final decision: {final}")
        if final == "escalate_access":
            print(f"  ESCALATION: {r.topic}: {results[i].coverage_impact}")
        if final == "stop_workflow":                 # the re-dispatch itself hit billing: stop here too
            print("  billing/quota failure during re-dispatch: no further model calls")
            break
    return results


ARCHIVE = InjectedArchive()
with capture_spans() as collector:
    with trace("Creative industries report (offline)", group_id="ch10-mini-project"):
        results = await coordinate(live=False)

print()
print(render_report("AI impact on creative industries", results))
print("\nBackend calls:", dict(ARCHIVE.calls))
print("Trace summary (errors only):")
for span in collector.spans:
    if span.error:
        print(f"  {span.span_data.type:<8} {getattr(span.span_data, 'name', '') or '':<26} {span.error['message']}")
print("Too many gaps?", too_many_gaps(results))

> **Observe:**
> - The visual-art `503` and the first music blip never reach the report: they were fixed locally (runner retry, tool retry).
> - The vocal-synthesis timeout cost a bounded number of backend calls, not an open-ended loop: 8 archive calls = 4 searches × 2 tool attempts (the generation search that hit the blip, the planned vocal-synthesis search, the worker's one narrower retry, and the coordinator's one re-dispatch). The music section kept its finding, and its coverage note keeps both attempts: `merge` recomputes the status from the original gap and the re-dispatch instead of letting either one overwrite the other.
> - The paywall went straight to escalation, no retries. The empty literature search counts as coverage.
> - The trace shows every handled tool error (function spans) and every partial worker (your custom spans), although no run raised.
> - Two of three sections have gaps, so `too_many_gaps` is `True`: publish this as a clearly marked partial report, or ask the user whether to continue.
>
> **Try it:** (1) make the music re-dispatch plan search `"AI singing-voice tools in trade press"` (no "vocal") and watch the section become FULL while keeping both findings (the original one and the trade-press review). (2) Inject `billing_429()` as the literature worker's first model step and check that the coordinator stops the workflow **before** re-dispatching music: stop conditions are scanned across all results before any new work starts.

In [ ]:
# (live) ~ $0.03. The same coordinator on OPENAI_MODEL, with the same fault-injected tools.
# Tracing is exported to the Traces dashboard (look for group_id "ch10-mini-project-live").
if not os.getenv("OPENAI_API_KEY"):
    print("Skipping the live run: set OPENAI_API_KEY in .env")
else:
    ARCHIVE = InjectedArchive()
    with trace("Creative industries report (live)", group_id="ch10-mini-project-live"):
        live_results = await coordinate(live=True)
    print()
    print(render_report("AI impact on creative industries (live)", live_results))
    print("\nBackend calls:", dict(ARCHIVE.calls))

> **Observe (live):** compare the live report with the offline one. A real model may plan different queries, but the injected faults still surface as the same structured statuses because tools, wrapper, decision rules and report renderer are all code.

## Recap

- **Classify by `error.code`, not status:** `429 slow_down` backs off; `credit_balance_exhausted` and spend/usage-limit codes stop and alert.
- The `openai` client **already retried twice** (status-based, `Retry-After` ≤ 120 s, `x-should-retry`). Pick one owning retry layer with a deadline; nested retries multiply.
- Agents SDK: `MaxTurnsExceeded` and friends carry `run_data`; `error_handlers` turn three failure kinds into controlled output; `ModelRetrySettings` is opt-in, and a status-based policy needs a billing veto.
- Tools return **errors as data** via `failure_error_function`; the default message is generic, `None` fails the run, raw exception text leaks internals. Tool `timeout` needs `async`.
- Workers always return a `WorkerResult` with partial findings; use `gather` or wrappers, never `TaskGroup`, for independent workers; give `as_tool` an extractor and a failure formatter.
- Recover locally first, then let the coordinator re-dispatch **once**, escalate, or accept partial; compute coverage in code and verify model claims with a guardrail.
- Traces record handled tool errors on function spans; add `custom_span` + `set_error` per worker; background responses and Agents API turns end in terminal statuses you must classify.

Next: [homework](02_homework.ipynb)